In [2]:
from langgraph.graph import StateGraph

In [4]:
help(StateGraph)

Help on class StateGraph in module langgraph.graph.state:

class StateGraph(typing.Generic)
 |  StateGraph(state_schema: 'type[StateT]', context_schema: 'type[ContextT] | None' = None, *, input_schema: 'type[InputT] | None' = None, output_schema: 'type[OutputT] | None' = None, **kwargs: 'Unpack[DeprecatedKwargs]') -> 'None'
 |
 |  A graph whose nodes communicate by reading and writing to a shared state.
 |
 |  The signature of each node is `State -> Partial<State>`.
 |
 |  Each state key can optionally be annotated with a reducer function that
 |  will be used to aggregate the values of that key received from multiple nodes.
 |  The signature of a reducer function is `(Value, Value) -> Value`.
 |
 |  !!! warning
 |
 |      `StateGraph` is a builder class and cannot be used directly for execution.
 |      You must first call `.compile()` to create an executable graph that supports
 |      methods like `invoke()`, `stream()`, `astream()`, and `ainvoke()`. See the
 |      `CompiledStateGr

In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_ext.models.ollama import OllamaChatCompletionClient

# Define a model client. You can use other model client that implements
# the `ChatCompletionClient` interface.
model_client = OllamaChatCompletionClient(
    model="qwen3:8b",
    # api_key="YOUR_API_KEY",
)


# Define a simple function tool that the agent can use.
# For this example, we use a fake weather tool for demonstration purposes.
async def get_weather(city: str) -> str:
    """Get the weather for a given city."""
    return f"The weather in {city} is 73 degrees and Sunny."


# Define an AssistantAgent with the model, tool, system message, and reflection enabled.
# The system message instructs the agent via natural language.
agent = AssistantAgent(
    name="weather_agent",
    model_client=model_client,
    tools=[get_weather],
    system_message="You are a helpful assistant.",
    reflect_on_tool_use=True,
    model_client_stream=True,  # Enable streaming tokens from the model client.
)


# Run the agent and stream the messages to the console.
async def main() -> None:
    await Console(agent.run_stream(task="What is the weather in New York?"))
    # Close the connection to the model client.
    await model_client.close()


# NOTE: if running this inside a Python script you'll need to use asyncio.run(main()).
await main()


---------- TextMessage (user) ----------
What is the weather in New York?


ConnectError: All connection attempts failed